In [6]:
import os
import math
import random
import torch
from torch.utils.data import IterableDataset
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    BertTokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    BloomForCausalLM,
    AutoConfig,
)
from peft import LoraConfig, get_peft_model

In [ ]:
import time
import torch.nn as nn
from approx.he_approx import HEGELU, HELayerNorm

BENCHMARK_ONLY = True          # 只跑计时，不跑Trainer
USE_HE_APPROX = True           # True: 替换; False: baseline
REPLACE_LN = True
REPLACE_GELU = True

BENCH_BS = 4                   # 建议对齐你的训练 batch: 4
BENCH_SEQ = 1024               # 对齐 block_size
WARMUP_ITERS = 30
BENCH_ITERS = 100              # 100~300 一般足够稳定

USE_HE_APPROX = os.environ.get("USE_HE_APPROX", "0") == "1"
REPLACE_LN = os.environ.get("REPLACE_LN", "1") == "1"
REPLACE_GELU = os.environ.get("REPLACE_GELU", "1") == "1"

BENCH_STEPS = int(os.environ.get("BENCH_STEPS", "200"))   # 200 步通常 10~20 分钟级（视GPU）
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "./ft_out_bench")
RESUME_CKPT = os.environ.get("RESUME_CKPT", "")           # 为空则不resume


In [ ]:
def apply_he_approx(model, gelu_approx="poly", ln_approx="affine_only"):
    # 递归替换 LayerNorm
    def _replace_ln(parent: nn.Module):
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.LayerNorm):
                new_ln = HELayerNorm(
                    child.normalized_shape,
                    eps=child.eps,
                    elementwise_affine=child.elementwise_affine,
                    approx=ln_approx,
                )
                # 复制原 LN 的 gamma/beta，避免行为突变
                if child.elementwise_affine:
                    new_ln.weight.data.copy_(child.weight.data)
                    new_ln.bias.data.copy_(child.bias.data)
                setattr(parent, name, new_ln)
            else:
                _replace_ln(child)

    # 递归替换 GELU：两类情况
    # 1) 模块型 nn.GELU
    # 2) GPT2MLP 常见的 self.act 是函数（callable），可直接替换成 HEGELU 模块
    def _replace_gelu(parent: nn.Module):
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.GELU):
                setattr(parent, name, HEGELU(approx=gelu_approx))
            else:
                _replace_gelu(child)

        # 处理 MLP 内部 act（通常不是 nn.Module）
        if parent.__class__.__name__.lower().endswith("mlp"):
            if hasattr(parent, "act") and callable(getattr(parent, "act")) and not isinstance(getattr(parent, "act"), nn.Module):
                setattr(parent, "act", HEGELU(approx=gelu_approx))

    if REPLACE_LN:
        _replace_ln(model)
    if REPLACE_GELU:
        _replace_gelu(model)

    return model

In [7]:
raw = load_dataset("text", data_files={"train": "./data/CLUECorpusSmall.txt"})["train"]
raw = raw.train_test_split(test_size=0.01, seed=42)
holdout = raw["test"].shuffle(seed=42)

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

In [8]:
raw_demo = DatasetDict({
    "train": raw["train"].select(range(2000)),
    "validation": holdout.select(range(200)),
    "test": holdout.select(range(200, 200 + 200)),  # 也可扩大
})

In [9]:
raw_demo['train'][100]

{'text': '在美国黑人区生活是一种什么样的体验？看大家都挺感兴趣的，我就再讲讲。我去某地黑人区不是图房租便宜，或者做个大死之类的。实际上再作死的人也是怕死的。我在美国被劫犯用上膛的枪指过头，在那之前我一直以为自己对这方面肯定不怕。但那一瞬间还是慌，对方动一动手指你就能死的时候，是真的慌。我去黑人区主要是在那里做过一段时间的志愿者，社区工作之类的。对口的是那边一个教会开的学校，初中。美国人均GDP五万多美金，全世界人均都可以排进前十。'}

In [10]:
tokenizer = BertTokenizer.from_pretrained("uer/gpt2-distil-chinese-cluecorpussmall")
model = GPT2LMHeadModel.from_pretrained("uer/gpt2-distil-chinese-cluecorpussmall")

In [12]:
cls_id = tokenizer.cls_token_id
sep_id = tokenizer.sep_token_id

block_size = 1024
# 如果你希望每个块都形如 [CLS] ... [SEP]，那中间内容最多 block_size-2
max_content_len = block_size - 2

# 可选：滑窗重叠步长。0 表示不重叠（推荐先从 0 开始跑通）
stride = 0
step = max_content_len if stride == 0 else max_content_len - stride

def tokenize_lines(examples):
    return tokenizer(examples["text"], add_special_tokens=False)

tok = raw_demo.map(tokenize_lines, batched=True, remove_columns=["text"])

def split_to_chunks(examples):
    input_ids_out, attn_out, labels_out = [], [], []
    for ids in examples["input_ids"]:
        if not ids:
            continue

        start = 0
        while start < len(ids):
            chunk = ids[start : start + max_content_len]
            # 组装成 [CLS] chunk [SEP]
            chunk_ids = [cls_id] + chunk + [sep_id]
            input_ids_out.append(chunk_ids)
            attn_out.append([1] * len(chunk_ids))
            labels_out.append(chunk_ids.copy())

            if start + max_content_len >= len(ids):
                break
            start += step

    return {"input_ids": input_ids_out, "attention_mask": attn_out, "labels": labels_out}

lm_demo_ds = tok.map(split_to_chunks, batched=True, remove_columns=tok["train"].column_names)
save_path = "./data/lm_demo_ds"
lm_demo_ds.save_to_disk(save_path)
print("saved to:", save_path)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2188 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/227 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/215 [00:00<?, ? examples/s]

saved to: ./data/lm_demo_ds


In [16]:
raw_demo

print(f"训练集最大字符长度: {max_char_length:,}")

训练集最大字符长度: 10


In [ ]:
# 确保 pad_token_id 存在
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# 2) （可选）LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"],
)
model = get_peft_model(model, lora_config)
if USE_HE_APPROX:
    model = apply_he_approx(model, gelu_approx="poly", ln_approx="affine_only")

model.print_trainable_parameters()

# 3) 关键：把 labels 列移除（让 collator 负责生成 labels + padding=-100）
lm_ds = lm_demo_ds.remove_columns(["labels"])

# 4) collator：CausalLM 训练必须 mlm=False
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5) TrainingArguments（先跑通，后面再加大 max_steps）
args = TrainingArguments(
    output_dir="./ft_out",
    overwrite_output_dir=True,
    # Batch size设置
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    # 学习率与优化
    learning_rate=2e-4,
    max_steps=50,
    num_train_epochs=1,
    # 热身步骤需要大幅增加（对于1600万数据）
    warmup_steps=10,
    # 大幅增加日志间隔（从20到10000步）
    logging_steps=10,  # 每1万步打印一次（不是10万，这样可以看到进展）
    # 评估和保存间隔调整为每10万步
    eval_strategy="steps",
    eval_steps=10,  # 每10万步评估一次
    save_strategy="steps",
    save_steps=10,  # 每10万步保存一次
    save_total_limit=10,  # 保存5个检查点（50万步时会有5个）
    # 其他设置
    fp16=torch.cuda.is_available(),
    report_to="tensorboard",  # 建议使用tensorboard记录
    seed=42,
    # 新增优化参数
    optim="adamw_torch",
    lr_scheduler_type="cosine",  # 余弦退火学习率
    weight_decay=0.01,  # 权重衰减防止过拟合
    gradient_checkpointing=False,  # 节省显存
    dataloader_num_workers=4,  # 加速数据加载
    group_by_length=True,  # 按长度分组提高效率
)

# 6) Trainer
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=lm_ds["train"],
    eval_dataset=lm_ds["validation"],
    data_collator=data_collator,
)

trainer.train()

# 训练结束后：验证集（如果你想再确认一次）
val_metrics = trainer.evaluate()  # 使用 Trainer 里默认的 eval_dataset
val_loss = val_metrics["eval_loss"]
val_ppl = math.exp(val_loss) if val_loss < 20 else float("inf")
print(f"[VAL] loss={val_loss:.4f}, ppl={val_ppl:.2f}")

# 最终测试集：只做一次
test_metrics = trainer.evaluate(eval_dataset=lm_ds["test"])
test_loss = test_metrics["eval_loss"]
test_ppl = math.exp(test_loss) if test_loss < 20 else float("inf")
print(f"[TEST] loss={test_loss:.4f}, ppl={test_ppl:.2f}")

test_metrics
# 保存 LoRA adapter
model.save_pretrained("./ft_out/lora_adapter")
tokenizer.save_pretrained("./ft_out/lora_adapter")


In [ ]:
def generate_with_device_fix(model, tokenizer, prompt, max_length=50, **kwargs):
    """自动处理设备问题的生成函数"""

    # 确保模型在评估模式
    model.eval()

    # 编码输入
    inputs = tokenizer(prompt, return_tensors="pt")

    # 获取模型设备，并将输入移到相同设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}


    print(f"输入在设备: {inputs['input_ids'].device}")
    print(f"模型在设备: {device}")

    # 生成参数
    gen_kwargs = {
        'max_length': 100,
        'do_sample': True,
        'temperature': 0.7,
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'repetition_penalty': 1.1,
    }
    gen_kwargs.update(kwargs)

    # 生成
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs.get('attention_mask', None),
            **gen_kwargs
        )

    # 解码
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated

# 测试修复后的生成
test_prompt = "今天天气"
try:
    generated_text = generate_with_device_fix(model, tokenizer, test_prompt, max_length=30)
    print(f"\n测试生成:")
    print(f"输入: {test_prompt}")
    print(f"输出: {generated_text}")
except Exception as e:
    print(f"生成失败: {e}")

In [4]:
%load_ext tensorboard
%tensorboard --logdir ./ft_out/runs
%reload_ext tensorboard
%tensorboard --logdir ./ft_out/runs --port 6006

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6008 (pid 21822), started 0:02:41 ago. (Use '!kill 21822' to kill it.)

ERROR: Failed to launch TensorBoard (exited with 255).
Contents of stderr:
/home/ubuntu24-xyx/anaconda3/envs/GPT2_Crypt/lib/python3.10/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

E0108 12:30:00.186166 135353970288448 program.py:300] TensorBoard could not bind to port 6006, it was already in use
ERROR: TensorBoard could not bind to port 6006, it was already in use